# Krillion Solver: fine-tune on your answers

Trains your model on the answers the game accepted, then packages it for Ollama.

1. **Runtime → Change runtime type → T4 GPU**, then run the cells in order.
2. When asked, upload `training_examples.jsonl` (made by **Answer history → Export training data** in the app).
3. The last cell downloads `krillion-solver-model.zip`. Then follow `training/README.md` on your PC.

This notebook wasn't run before release (it needs a GPU). If a cell fails, the error usually
names a renamed setting in a newer library version.

In [ ]:
%%capture
!pip install unsloth

In [ ]:
# Settings. Match the base model to the one you use in the app:
#   llama3.2:3b -> "unsloth/Llama-3.2-3B-Instruct"   (trains in a few minutes on a T4)
#   llama3.1:8b -> "unsloth/Meta-Llama-3.1-8B-Instruct"
BASE_MODEL = "unsloth/Llama-3.2-3B-Instruct"
EPOCHS = 3

In [ ]:
import json
from google.colab import files

uploaded = files.upload()  # choose training_examples.jsonl
name = next(n for n in uploaded if n.endswith(".jsonl"))
rows = [json.loads(line) for line in open(name, encoding="utf-8") if line.strip()]
print(len(rows), "training examples")
assert rows, "No examples: mark some answers as accepted in the app first"

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(BASE_MODEL, max_seq_length=2048, load_in_4bit=True)
model = FastLanguageModel.get_peft_model(
    model, r=16, lora_alpha=16, lora_dropout=0, bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth", random_state=0)

In [ ]:
from datasets import Dataset

# Same chat format the app sends: system prompt, the prompt, and a JSON list of answers.
dataset = Dataset.from_list(rows).map(
    lambda row: {"text": tokenizer.apply_chat_template(row["messages"], tokenize=False)},
    remove_columns=["messages"])
print(dataset[0]["text"][:600])

In [ ]:
import inspect
from trl import SFTConfig, SFTTrainer

# Setting names differ between trl versions; pass only the ones this version knows.
wanted = dict(dataset_text_field="text", max_seq_length=2048, max_length=2048,
              per_device_train_batch_size=2, gradient_accumulation_steps=4, num_train_epochs=EPOCHS,
              learning_rate=2e-4, logging_steps=5, output_dir="outputs", report_to="none", seed=0)
known = inspect.signature(SFTConfig).parameters
config = SFTConfig(**{k: v for k, v in wanted.items() if k in known})
trainer_args = dict(model=model, train_dataset=dataset, args=config)
if "processing_class" in inspect.signature(SFTTrainer).parameters:
    trainer_args["processing_class"] = tokenizer
else:
    trainer_args["tokenizer"] = tokenizer
trainer = SFTTrainer(**trainer_args)
trainer.train()

In [ ]:
# Quick check: does it still answer in the app's JSON format?
FastLanguageModel.for_inference(model)
messages = rows[0]["messages"][:2]
inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt").to("cuda")
out = model.generate(input_ids=inputs, max_new_tokens=200, temperature=0.7, do_sample=True)
print(tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True))

In [ ]:
import glob, os, shutil

# 4-bit GGUF, the format Ollama runs (about 2 GB for 3B, 5 GB for 8B).
model.save_pretrained_gguf("krillion_gguf", tokenizer, quantization_method="q4_k_m")
ggufs = sorted(glob.glob("**/*.gguf", recursive=True), key=lambda p: ("q4_k_m" not in p.lower(), p))
print(ggufs)
assert ggufs, "No GGUF file was written; check the output above"

os.makedirs("export", exist_ok=True)
shutil.copy(ggufs[0], "export/krillion-solver.gguf")
# Llama 3 chat template, so Ollama sends system + user turns the way the model was trained.
open("export/Modelfile", "w").write('''FROM ./krillion-solver.gguf
TEMPLATE """{{ if .System }}<|start_header_id|>system<|end_header_id|>

{{ .System }}<|eot_id|>{{ end }}{{ if .Prompt }}<|start_header_id|>user<|end_header_id|>

{{ .Prompt }}<|eot_id|>{{ end }}<|start_header_id|>assistant<|end_header_id|>

{{ .Response }}<|eot_id|>"""
PARAMETER stop "<|eot_id|>"
PARAMETER stop "<|start_header_id|>"
PARAMETER stop "<|end_header_id|>"
PARAMETER num_ctx 2048
''')
shutil.make_archive("krillion-solver-model", "zip", "export")
files.download("krillion-solver-model.zip")